# Aggregate Economic Loss Simulator – Extended Project (Solution)

**Practical Economics / Risk-Management application of the Monte-Carlo idea from Al Sweigart’s Million Dice Roll Statistics Simulator (Project #46)**

In insurance, banking and portfolio risk management we frequently need the distribution of the *sum* of many independent random losses (or returns).  
When each individual risk is discrete (or discretized), the exact convolution quickly becomes expensive. Monte Carlo simulation is the standard practical tool.

This notebook extends the classic “roll N dice one million times” idea into:

- Simulation of **aggregate portfolio / claim loss** from N independent discrete risks
- Progress reporting, clean frequency table and histogram
- Empirical **Value-at-Risk (VaR)** and **Expected Shortfall (ES)**
- Two alternate implementations (Counter + NumPy vectorized)
- Exact theoretical distribution via dynamic programming
- Parameter sweeps that illustrate the Central Limit Theorem in a risk context
- Configurable “More Practice” and Simulation sections

Use the matching **Practice Skeleton** to implement the pieces yourself first.


In [ ]:
from IPython.display import Image, display
display(Image(filename='aggregate_economic_loss_flowchart.png', width=950))
print("Flowchart: Aggregate Economic Loss Simulator (Monte Carlo path + alternates).")


## 0. Imports & Reproducibility

We keep the same libraries as the original dice project so the code patterns stay familiar.


In [ ]:
import random
import time
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

# Educational reproducibility (comment out for pure randomness)
random.seed(42)
np.random.seed(42)

print("Libraries loaded.")


## 1. Core Simulation Function

`simulate_aggregate_loss(n_risks, sides=6, num_simulations=1_000_000, show_progress=True)`

Each of the `n_risks` independent exposures produces a discrete loss drawn uniformly from 1 … `sides` (exactly analogous to a die).  
The function returns a dictionary mapping every possible aggregate loss to its frequency count.

In a real risk model the support would be the possible claim sizes or P&L levels; the uniform discrete case keeps the parallel with the original book project crystal clear while still being a valid (if stylized) economic example.


In [ ]:
def simulate_aggregate_loss(n_risks, sides=6, num_simulations=1_000_000, show_progress=True):
    """Simulate the sum of n_risks independent discrete losses (each 1..sides).
    Returns dict: aggregate_loss -> count.
    """
    min_loss = n_risks * 1
    max_loss = n_risks * sides
    results = {total: 0 for total in range(min_loss, max_loss + 1)}

    if show_progress:
        print(f'Simulating {num_simulations:,} scenarios of {n_risks} independent risks '
              f'(each loss ∈ {{1..{sides}}})...')
    last_print_time = time.time()

    for i in range(num_simulations):
        if show_progress and time.time() > last_print_time + 1:
            pct = round(i / (num_simulations / 100), 1)
            print(f'{pct}% done...')
            last_print_time = time.time()

        total_loss = 0
        for _ in range(n_risks):
            total_loss += random.randint(1, sides)
        results[total_loss] += 1

    return results


# Demo with a small number of simulations for notebook speed
demo = simulate_aggregate_loss(n_risks=2, sides=6, num_simulations=100_000, show_progress=True)
print("Sample counts (first few):", dict(list(demo.items())[:5]), "...")


## 2. Display Frequency Table + Basic Risk Metrics

Print the classic TOTAL – COUNT – PERCENTAGE table and, in addition, compute two key risk-management numbers from the empirical distribution:

- **VaR_α** (Value-at-Risk) – the α-quantile of the loss distribution  
- **Expected Shortfall (ES_α)** – average loss in the worst (1-α) tail


In [ ]:
def display_results_and_risk_metrics(results, num_simulations=1_000_000, alpha=0.95):
    """Print frequency table and compute empirical VaR / Expected Shortfall."""
    print('AGGREGATE LOSS - COUNT - PERCENTAGE')
    for loss in sorted(results.keys()):
        count = results[loss]
        pct = round(count / num_simulations * 100, 1)
        print(f' {loss:3d} - {count:7d} - {pct:5.1f}%')

    # Build sorted list of all simulated losses (for quantiles)
    # For large num_simulations we work from the histogram to stay memory-friendly
    sorted_losses = []
    for loss in sorted(results.keys()):
        sorted_losses.extend([loss] * results[loss])

    # Empirical VaR (quantile)
    idx = int(np.ceil(alpha * num_simulations)) - 1
    var = sorted_losses[idx]

    # Expected Shortfall = mean of losses beyond VaR
    tail = sorted_losses[idx:]
    es = np.mean(tail) if tail else var

    print(f'\nEmpirical risk metrics (α = {alpha}):')
    print(f'  VaR_{alpha:.0%}  = {var}')
    print(f'  ES_{alpha:.0%}   = {es:.2f}')
    return var, es


print("=== 2 independent risks, 100 000 scenarios ===")
var_demo, es_demo = display_results_and_risk_metrics(demo, 100_000, alpha=0.95)


## 3. Visualization – Aggregate Loss Distribution

A histogram (bar chart) of the empirical percentages makes the shape of the portfolio-loss distribution immediate.  
For a small number of risks the distribution is triangular / skewed; as the number of risks grows it becomes approximately normal (Central Limit Theorem – the theoretical foundation of many regulatory capital models).


In [ ]:
def plot_loss_distribution(results, n_risks, sides, num_simulations, title_suffix="", fname='econ_loss_distribution.png'):
    losses = sorted(results.keys())
    percentages = [results[l] / num_simulations * 100 for l in losses]

    plt.figure(figsize=(10, 5))
    plt.bar(losses, percentages, color='steelblue', edgecolor='navy', alpha=0.85)
    plt.xlabel('Aggregate Loss', fontsize=12)
    plt.ylabel('Percentage (%)', fontsize=12)
    plt.title(f'Empirical aggregate-loss distribution\n'
              f'{n_risks} independent risks × discrete support 1..{sides} '
              f'({num_simulations:,} Monte-Carlo scenarios){title_suffix}', fontsize=13)
    plt.xticks(losses[::max(1, len(losses)//15)])
    plt.grid(axis='y', alpha=0.3)

    mode = max(results, key=results.get)
    plt.annotate(f'mode={mode}', xy=(mode, results[mode]/num_simulations*100),
                 xytext=(mode + 0.8, results[mode]/num_simulations*100 + 1.2),
                 arrowprops=dict(arrowstyle='->', color='crimson'), color='crimson')
    plt.tight_layout()
    plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Chart saved as {fname}")


plot_loss_distribution(demo, 2, 6, 100_000)


## 4. Alternate Implementation #1 – collections.Counter

Functionally identical, slightly more compact; useful when the support is not known in advance.


In [ ]:
def simulate_with_counter(n_risks, sides=6, num_simulations=1_000_000):
    counter = Counter()
    for _ in range(num_simulations):
        total = sum(random.randint(1, sides) for _ in range(n_risks))
        counter[total] += 1
    # Guarantee every possible key appears
    for t in range(n_risks, n_risks * sides + 1):
        counter.setdefault(t, 0)
    return dict(counter)


c_res = simulate_with_counter(2, 6, 50_000)
print("Counter version (50k scenarios) – first few keys:", dict(list(c_res.items())[:4]))
print("Total scenarios accounted for:", sum(c_res.values()))


## 5. Alternate Implementation #2 – NumPy Vectorized (production speed)

Generate the entire (num_simulations × n_risks) matrix at once and sum along the risk axis.  
This is the style used in real risk-engine code when the number of scenarios reaches millions.


In [ ]:
def simulate_numpy(n_risks, sides=6, num_simulations=1_000_000):
    rolls = np.random.randint(1, sides + 1, size=(num_simulations, n_risks))
    totals = rolls.sum(axis=1)
    unique, counts = np.unique(totals, return_counts=True)
    results = {int(u): int(c) for u, c in zip(unique, counts)}
    for t in range(n_risks, n_risks * sides + 1):
        results.setdefault(t, 0)
    return results


# Speed comparison
t0 = time.time()
_ = simulate_aggregate_loss(5, 6, 200_000, show_progress=False)
t_loop = time.time() - t0
t0 = time.time()
_ = simulate_numpy(5, 6, 200_000)
t_np = time.time() - t0
print(f"Pure-Python loop 200k scenarios of 5 risks: {t_loop:.3f}s")
print(f"NumPy vectorized               : {t_np:.3f}s")
print(f"Speed-up ≈ {t_loop / max(t_np, 1e-9):.1f}×")


## 6. Exact Theoretical Distribution (Dynamic Programming)

When each risk is discrete uniform on {1…S} the exact probability mass function of the sum can be obtained by the classic recurrence used for dice:

`dp[k][s] = number of ways to obtain aggregate loss s with k risks`.

This lets risk managers benchmark the Monte-Carlo approximation against the true distribution for moderate N.


In [ ]:
def theoretical_probs(n_risks, sides=6):
    max_sum = n_risks * sides
    dp = [0] * (max_sum + 1)
    dp[0] = 1
    for k in range(1, n_risks + 1):
        new_dp = [0] * (max_sum + 1)
        for prev, ways in enumerate(dp):
            if ways == 0:
                continue
            for face in range(1, sides + 1):
                new_dp[prev + face] += ways
        dp = new_dp
    total_ways = sides ** n_risks
    return {s: dp[s] / total_ways for s in range(n_risks, max_sum + 1) if dp[s] > 0}


theo = theoretical_probs(2, 6)
print("Exact probabilities for 2 independent risks (support 1..6):")
for loss, p in sorted(theo.items()):
    print(f"  loss={loss}: {p*100:.1f}%  ({int(round(p*36))}/36)")


## 7. Empirical vs Theoretical Comparison + VaR Check

Run a larger Monte-Carlo experiment and overlay the exact PMF.  
Also verify that the empirical VaR converges to the theoretical quantile.


In [ ]:
def compare_emp_theo(n_risks=2, sides=6, num_simulations=500_000, alpha=0.95):
    emp = simulate_numpy(n_risks, sides, num_simulations)
    theo = theoretical_probs(n_risks, sides)

    losses = sorted(emp.keys())
    emp_pct = [emp[l] / num_simulations * 100 for l in losses]
    theo_pct = [theo.get(l, 0) * 100 for l in losses]

    plt.figure(figsize=(10, 5))
    width = 0.4
    x = np.array(losses)
    plt.bar(x - width/2, emp_pct, width, label='Empirical (Monte-Carlo)', color='steelblue', alpha=0.8)
    plt.bar(x + width/2, theo_pct, width, label='Theoretical (exact)', color='darkorange', alpha=0.8)
    plt.xlabel('Aggregate Loss')
    plt.ylabel('Percentage (%)')
    plt.title(f'{n_risks} independent risks: Empirical ({num_simulations:,}) vs Exact')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('econ_loss_emp_vs_theo.png', dpi=120, bbox_inches='tight')
    plt.show()

    max_err = max(abs(e - t) for e, t in zip(emp_pct, theo_pct))
    print(f"Max absolute percentage error: {max_err:.3f} pp")

    # Empirical VaR from the same run
    sorted_losses = []
    for l in losses:
        sorted_losses.extend([l] * emp[l])
    idx = int(np.ceil(alpha * num_simulations)) - 1
    emp_var = sorted_losses[idx]
    print(f"Empirical VaR_{alpha:.0%} from this run: {emp_var}")


compare_emp_theo(2, 6, 500_000)


## 8. More Practice Exercises

1. Simulate 200 000 scenarios of **5 independent risks** (support 1..6) and report the mode and VaR_95%.  
2. Change the support to a more realistic asymmetric loss set, e.g. possible losses {0, 1, 2, 5, 10} (you will need a small modification of the sampling line).  
3. Model **10 independent projects** each with a binary outcome (success = +1, failure = 0) and examine the distribution of the number of successful projects.  
4. Use the theoretical function to confirm that two independent risks with support {1,2} give the classic 25 % / 50 % / 25 % distribution.


In [ ]:
# Practice 1 – 5 risks
print("=== Practice 1: 5 independent risks (200 k scenarios) ===")
res5 = simulate_numpy(5, 6, 200_000)
display_results_and_risk_metrics(res5, 200_000, alpha=0.95)
mode5 = max(res5, key=res5.get)
print(f"Mode ≈ {mode5}")

# Practice 3 – binary outcomes (10 projects)
print("\n=== Practice 3: 10 binary projects (success=1 / failure=0) ===")
# sides=2 with faces {1,2} maps to {0,1} after a trivial shift; we just use 1..2 and interpret
res_bin = simulate_numpy(10, 2, 100_000)
display_results_and_risk_metrics(res_bin, 100_000, alpha=0.95)

# Practice 4 – theoretical two binary risks
print("\n=== Practice 4: exact distribution of two binary risks ===")
theo_bin = theoretical_probs(2, 2)
print({k: f'{v*100:.1f}%' for k, v in theo_bin.items()})


## 9. Simulation Section – Parameter Sweeps & CLT in Risk Management

Risk managers care about how the shape of the aggregate-loss distribution changes when the number of independent risks grows.  
By the Central Limit Theorem the standardized sum converges to a normal distribution – this is why many regulatory formulas can use normal or log-normal approximations for large portfolios.

Edit the list of configurations below and re-run to explore different regimes.


In [ ]:
def run_risk_sweep(configs):
    """configs = list of (n_risks, sides, n_sims, label)"""
    fig, axes = plt.subplots(1, len(configs), figsize=(5*len(configs), 4), sharey=True)
    if len(configs) == 1:
        axes = [axes]

    for ax, (n, s, sims, label) in zip(axes, configs):
        res = simulate_numpy(n, s, sims)
        losses = sorted(res.keys())
        pct = [res[l]/sims*100 for l in losses]
        ax.bar(losses, pct, color='teal', alpha=0.8, edgecolor='darkgreen')
        mean = sum(l * res[l] for l in losses) / sims
        ax.axvline(mean, color='red', linestyle='--', label=f'mean≈{mean:.1f}')
        ax.set_title(label)
        ax.set_xlabel('Aggregate Loss')
        ax.legend(fontsize=8)
        ax.grid(axis='y', alpha=0.3)

    axes[0].set_ylabel('Percentage (%)')
    plt.suptitle('Aggregate-loss distributions – Central Limit Theorem in action', fontsize=14)
    plt.tight_layout()
    plt.savefig('econ_loss_simulation_sweep.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Sweep chart saved as econ_loss_simulation_sweep.png")


experiments = [
    (1, 6, 50_000, '1 risk (flat)'),
    (2, 6, 100_000, '2 risks (triangle)'),
    (5, 6, 200_000, '5 risks (≈normal)'),
    (15, 6, 200_000, '15 risks (very normal)'),
]
run_risk_sweep(experiments)


## 10. Full-Scale Classic-Style Run (optional)

Uncomment the cell below to run a full 1 000 000-scenario simulation of a small portfolio (2 risks).  
Progress messages appear once per second, exactly as in the original book program.


In [ ]:
# Uncomment for the authentic million-scenario run:
# full = simulate_aggregate_loss(2, sides=6, num_simulations=1_000_000, show_progress=True)
# display_results_and_risk_metrics(full, 1_000_000, alpha=0.99)
# plot_loss_distribution(full, 2, 6, 1_000_000, title_suffix=" – full 1 M scenarios")

print("Full-million cell is commented out for notebook responsiveness.")
print("Uncomment when you want the classic slow progress output.")


## Key Takeaways (Economics / Risk Management)

- Monte Carlo is the work-horse method for obtaining the distribution of aggregate losses when analytic convolution is impractical.
- Even a stylized discrete-uniform risk model already produces the classic shapes (triangular → normal) that risk managers rely on.
- Empirical VaR and Expected Shortfall are obtained directly from the ordered simulated losses – no closed-form assumption required.
- The Central Limit Theorem explains why large portfolios of independent risks can often be treated as approximately normal, justifying many regulatory capital formulas.
- Vectorized (NumPy) implementations turn a pedagogical loop into production-speed code while preserving the identical statistical meaning.
- Always benchmark Monte-Carlo results against an exact DP solution for small portfolios; the discrepancy quantifies sampling error.
